# ByteMe: Tokenizer Comparison

Compare 4 tokenization strategies on the same text:
- Byte-level BPE (GPT-4 style)
- BPE (standard)
- WordPiece (BERT)
- Unigram (T5/XLNet)

We test on regular text, code, emojis, multilingual, and URLs.

In [ ]:
import os
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from transformers import AutoTokenizer
import sentencepiece as spm
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter

## Test samples

We need text that covers different scenarios. Regular english, spanish, code, emojis, URLs, and mixed content. Each tokenizer will handle these differently.

In [ ]:
TEST_SAMPLES = {
    "english": "The quick brown fox jumps over the lazy dog. Natural language processing has transformed how we interact with computers.",
    "spanish": "El procesamiento de lenguaje natural ha transformado la forma en que interactuamos con las computadoras. Los modelos de lenguaje son cada vez mas potentes.",
    "code": "def fibonacci(n):\n    if n <= 1:\n        return n\n    return fibonacci(n-1) + fibonacci(n-2)\n\nresult = [fibonacci(i) for i in range(10)]",
    "emojis": "I love this movie! 🎬🍿 The acting was superb 👏👏👏 10/10 would recommend 🔥💯",
    "urls": "Check out https://huggingface.co/docs/transformers/main/tokenizer_summary and also visit http://arxiv.org/abs/1706.03762 for the original paper.",
    "mixed": "El modelo GPT-4 usa byte-level BPE con vocab de ~100k tokens. Mas info en https://openai.com 🤖",
    "rare_words": "The pneumonoultramicroscopicsilicovolcanoconiosis patient visited the otorhinolaryngologist for a consultation.",
    "numbers": "The model has 175,000,000,000 parameters and was trained on 300B tokens with a learning rate of 3e-4.",
}

print(f"Created {len(TEST_SAMPLES)} test samples")
for name, text in TEST_SAMPLES.items():
    print(f"  {name}: {len(text)} chars")

## Load tokenizers

We load one real tokenizer for each strategy. These are all pretrained and available from HuggingFace.

- **Byte-level BPE**: GPT-2 tokenizer (same family as GPT-4, just smaller vocab)
- **BPE**: GPT-NeoX / Pythia tokenizer
- **WordPiece**: BERT base uncased
- **Unigram**: T5 small (uses SentencePiece under the hood)

In [ ]:
TOKENIZERS = {
    "byte_bpe": {
        "name": "Byte-level BPE",
        "model": "gpt2",
        "example": "GPT-2, GPT-4, Llama",
    },
    "bpe": {
        "name": "BPE",
        "model": "EleutherAI/pythia-70m",
        "example": "Pythia, GPT-NeoX",
    },
    "wordpiece": {
        "name": "WordPiece",
        "model": "bert-base-uncased",
        "example": "BERT, DistilBERT",
    },
    "unigram": {
        "name": "Unigram",
        "model": "t5-small",
        "example": "T5, XLNet, ALBERT",
    },
}

loaded = {}
for key, info in TOKENIZERS.items():
    print(f"Loading {info['name']} ({info['model']})...")
    tok = AutoTokenizer.from_pretrained(info["model"])
    loaded[key] = tok
    print(f"  Vocab size: {tok.vocab_size:,}")

print(f"\nLoaded {len(loaded)} tokenizers")

## Tokenize everything

Run each tokenizer on each test sample. Track token count and the actual tokens produced.

In [ ]:
def tokenize_all(samples, tokenizers):
    """Tokenize all samples with all tokenizers. Returns a dict of results."""
    results = {}
    for sample_name, text in samples.items():
        results[sample_name] = {}
        for tok_key, tok in tokenizers.items():
            ids = tok.encode(text)
            tokens = tok.convert_ids_to_tokens(ids)
            results[sample_name][tok_key] = {
                "ids": ids,
                "tokens": tokens,
                "n_tokens": len(ids),
                "n_chars": len(text),
                "compression": len(text) / len(ids),  # chars per token
            }
    return results

results = tokenize_all(TEST_SAMPLES, loaded)

# quick summary
for sample_name, tok_results in results.items():
    print(f"\n{sample_name} ({tok_results[list(tok_results.keys())[0]]['n_chars']} chars):")
    for tok_key, r in tok_results.items():
        print(f"  {TOKENIZERS[tok_key]['name']:20s} -> {r['n_tokens']:4d} tokens (compression: {r['compression']:.1f} chars/tok)")

## Comparison table

Build a dataframe with token counts for every (sample, tokenizer) pair. This makes it easy to plot and compare.

In [ ]:
rows = []
for sample_name, tok_results in results.items():
    for tok_key, r in tok_results.items():
        rows.append({
            "sample": sample_name,
            "tokenizer": TOKENIZERS[tok_key]["name"],
            "n_tokens": r["n_tokens"],
            "n_chars": r["n_chars"],
            "compression": r["compression"],
        })

df = pd.DataFrame(rows)

# pivot table: samples as rows, tokenizers as columns
pivot = df.pivot(index="sample", columns="tokenizer", values="n_tokens")
pivot = pivot[["Byte-level BPE", "BPE", "WordPiece", "Unigram"]]  # fix column order
pivot

## Token count by sample type

Bar chart showing how many tokens each tokenizer produces per sample. Fewer tokens = better compression.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

pivot.plot(kind="bar", ax=ax, width=0.8)
ax.set_ylabel("Number of tokens")
ax.set_xlabel("")
ax.set_title("Token count by sample type and tokenizer")
ax.legend(title="Tokenizer", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("plots/token_counts.png", dpi=150, bbox_inches="tight")
plt.show()

## How each tokenizer splits the same sentence

This is the most revealing visualization. We take one sentence and show exactly how each tokenizer breaks it into tokens. Color-coded so you can see the boundaries.

In [ ]:
def show_token_splits(text, tokenizers, tokenizer_info):
    """Print how each tokenizer splits a text, with | as boundary markers."""
    print(f"Text: {text}\n")
    print("-" * 80)
    for tok_key, tok in tokenizers.items():
        ids = tok.encode(text)
        tokens = tok.convert_ids_to_tokens(ids)
        # clean up special characters in token display
        display_tokens = []
        for t in tokens:
            t = t.replace("Ġ", " ")   # GPT-2 space marker
            t = t.replace("▁", " ")   # SentencePiece space marker
            t = t.replace("##", "")   # WordPiece continuation marker
            display_tokens.append(t)
        name = tokenizer_info[tok_key]["name"]
        print(f"{name:20s} ({len(ids):3d} tokens): |{'|'.join(display_tokens)}|")
    print()

# test on a few interesting samples
for sample in ["english", "spanish", "code", "emojis", "rare_words"]:
    show_token_splits(TEST_SAMPLES[sample], loaded, TOKENIZERS)

## Compression ratio comparison

Compression ratio = characters per token. Higher means the tokenizer is more efficient (fewer tokens for the same text). This matters because transformer cost is O(n^2) in sequence length.

In [ ]:
pivot_comp = df.pivot(index="sample", columns="tokenizer", values="compression")
pivot_comp = pivot_comp[["Byte-level BPE", "BPE", "WordPiece", "Unigram"]]

fig, ax = plt.subplots(figsize=(14, 6))
pivot_comp.plot(kind="bar", ax=ax, width=0.8)
ax.set_ylabel("Characters per token (higher = better)")
ax.set_xlabel("")
ax.set_title("Compression ratio by sample type")
ax.axhline(y=4.0, color="gray", linestyle="--", alpha=0.5, label="4 chars/tok baseline")
ax.legend(title="Tokenizer", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("plots/compression_ratio.png", dpi=150, bbox_inches="tight")
plt.show()

## Vocab size vs coverage

Each tokenizer has a different vocab size. But bigger vocab does not always mean better compression. Lets look at the actual vocab sizes and what percentage of tokens are single characters (meaning the tokenizer had to fall back to character-level splitting).

In [ ]:
print("Vocab sizes:")
print("-" * 40)
for tok_key, tok in loaded.items():
    name = TOKENIZERS[tok_key]["name"]
    vocab_size = tok.vocab_size
    
    # count single-char tokens across all samples
    total_tokens = 0
    single_char_tokens = 0
    for sample_name in TEST_SAMPLES:
        tokens = results[sample_name][tok_key]["tokens"]
        total_tokens += len(tokens)
        for t in tokens:
            clean = t.replace("Ġ", "").replace("▁", "").replace("##", "")
            if len(clean) <= 1:
                single_char_tokens += 1
    
    pct = (single_char_tokens / total_tokens) * 100 if total_tokens > 0 else 0
    print(f"  {name:20s}  vocab: {vocab_size:>6,}  single-char fallback: {pct:.1f}%")

## OOV handling: what happens with unknown words?

The big difference between these tokenizers is how they handle words they have never seen.

- **Byte-level BPE**: Can always fall back to raw bytes. No UNK tokens ever. This is why GPT-2/4 can handle any input.
- **BPE**: Falls back to character level. Rare words get split into many small pieces.
- **WordPiece**: Uses the `[UNK]` token for truly unknown words, or splits with `##` prefix. BERT lowercases everything which helps but also loses info.
- **Unigram**: Falls back to single characters. SentencePiece always has a way to represent any input.

Lets test with some intentionally weird inputs.

In [ ]:
weird_inputs = {
    "made_up_word": "The xyzzyplugh algorithm outperforms blargenflarg in all benchmarks",
    "japanese": "自然言語処理は素晴らしい技術です",
    "arabic": "معالجة اللغة الطبيعية هي تقنية رائعة",
    "mixed_script": "Hello Привет こんにちは مرحبا 你好",
    "keyboard_smash": "asdfghjklqwertyuiopzxcvbnm!@#$%^&*()",
}

for name, text in weird_inputs.items():
    show_token_splits(text, loaded, TOKENIZERS)

## Heatmap: compression across all tests

A heatmap where darker = more tokens (worse compression). Makes it easy to spot which tokenizer struggles where.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

# normalize by chars so we compare fairly
pivot_norm = pivot.div(pivot.max(axis=1), axis=0)

im = ax.imshow(pivot_norm.values, cmap="YlOrRd", aspect="auto")
ax.set_xticks(range(len(pivot_norm.columns)))
ax.set_xticklabels(pivot_norm.columns, rotation=45, ha="right")
ax.set_yticks(range(len(pivot_norm.index)))
ax.set_yticklabels(pivot_norm.index)
ax.set_title("Relative token count (darker = more tokens = worse)")

# add text annotations
for i in range(len(pivot_norm.index)):
    for j in range(len(pivot_norm.columns)):
        val = pivot.values[i, j]
        ax.text(j, i, f"{int(val)}", ha="center", va="center", fontsize=9,
                color="white" if pivot_norm.values[i, j] > 0.7 else "black")

plt.colorbar(im, ax=ax, label="Relative token count")
plt.tight_layout()
plt.savefig("plots/heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## Key takeaways

**When does byte-level beat BPE?**

Byte-level BPE (GPT-2 style) handles everything thrown at it. No UNK tokens ever. It just works. The downside is slightly worse compression on normal english text because it uses bytes as the base unit instead of characters. But the robustness is worth it, which is why most modern LLMs use it.

**Which tokenizer handles code best?**

BPE and byte-level BPE are both decent at code because they can learn common patterns like `def`, `return`, `for i in range`. WordPiece (BERT) struggles more because it was trained on natural language, not code. Unigram falls in the middle.

**What about non-english languages?**

This is where the differences really show. WordPiece (BERT) was trained mostly on english and has a small vocab, so non-latin scripts get destroyed into tiny pieces. Byte-level BPE handles multilingual better because it can always fall back to bytes. Unigram with SentencePiece is also decent because SentencePiece is designed to be language-agnostic.

**Bottom line**

| If you need... | Use... |
|----------------|--------|
| Best all-around robustness | Byte-level BPE |
| Best english compression | BPE or WordPiece |
| Best multilingual support | Byte-level BPE or Unigram |
| Classification/NER on english | WordPiece (BERT ecosystem) |
| Sequence-to-sequence tasks | Unigram (T5 ecosystem) |

The industry has mostly converged on byte-level BPE for a reason. It is not the best at any single thing, but it is never bad at anything.